[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage3_text.ipynb)

# 阶段三 · 文本数据进阶（Day 12–15）

> 《Python 数据处理 20 天学习计划》阶段三配套教材。
> **开始前：文件 → 在云端硬盘中保存一份副本。**

**本阶段目标**：用 Pandas 的 `.str` 方法**批量**处理整列文本，会用正则表达式提取信息——阶段一处理的是「一个」字符串，本阶段处理「一百万个」。

## Day 12 · Pandas 向量化字符串操作（.str）

阶段一学过的 `strip / replace / split / upper` 等方法，在 Pandas 里加上 `.str` 前缀，就能**一次作用于整列**，不用写循环。

先造一份「脏」的联系人数据——真实世界的数据往往就长这样。

In [ ]:
import pandas as pd

contacts = pd.DataFrame({
    "姓名": ["  张三 ", "李 四", "王五", "赵六"],
    "城市": ["beijing", "SHANGHAI", "Guangzhou", "shenzhen"],
    "电话": ["138-0000-1111", "139 1111 2222", "137.2222.3333", "136-3333-4444"],
    "注册日期": ["2026-01-05", "2026/02/11", "2026.03.20", "2026-04-15"],
})
contacts

In [ ]:
# .str 常用操作：整列一次生效
print(contacts["姓名"].str.strip())        # 整列去两端空白
print(contacts["城市"].str.title())        # 整列首字母大写
print(contacts["城市"].str.upper())        # 整列全大写

In [ ]:
# str.contains：整列判断“是否包含”，常用来筛选
mask = contacts["电话"].str.contains("-")
contacts[mask]        # 电话里带 “-” 的行

In [ ]:
# str.replace：整列替换（默认按正则替换，普通替换建议加 regex=False）
contacts["城市"] = contacts["城市"].str.title()

# str.len：整列求长度；str.cat：列与列拼接
contacts["姓名长度"] = contacts["姓名"].str.strip().str.len()
contacts["标签"] = contacts["姓名"].str.strip().str.cat(contacts["城市"], sep=" @ ")
contacts

### ✏️ 练习 12-1
把「注册日期」列中的 `/` 和 `.` 统一替换成 `-`（提示：`str.replace` 连用两次，记得加 `regex=False`），结果存回「注册日期」列并查看整个表。

In [ ]:
# 练习 12-1：在这里写你的代码

## Day 13 · 正则表达式入门

正则表达式是「描述文本模式」的迷你语言。先记住 6 个最高频符号：

| 符号 | 含义 | 例子 |
|---|---|---|
| `\d` | 一个数字 | `\d\d\d` 匹配三位数字 |
| `\w` | 字母/数字/下划线 | — |
| `\s` | 空白字符 | — |
| `+` | 前面内容重复 1 次或多次 | `\d+` 一段连续数字 |
| `^` / `$` | 字符串开头 / 结尾 | `^2026` 以 2026 开头 |
| `[...]` | 字符集合 | `[/\.]` 斜杠或点 |

In [ ]:
import re   # Python 内置正则模块

text = "订单A12在2026年3月5日发货，订单B340在2026年4月18日发货"

# findall：找出所有匹配，返回列表
print(re.findall(r"\d+", text))           # 所有连续数字
print(re.findall(r"\d{4}年", text))        # 4位数字+“年”

# re.sub：把所有匹配替换成指定内容
print(re.sub(r"\d+", "N", text))           # 所有数字换成 N

In [ ]:
# 正则在 Pandas 里威力更大：str.replace / str.extract 都支持正则

# 把电话里的非数字字符全部去掉（\D 表示“非数字”）
contacts["电话"] = contacts["电话"].str.replace(r"\D", "", regex=True)

# extract：用括号（捕获组）把想要的部分“抓”出来——提取年份
contacts["注册年份"] = contacts["注册日期"].str.extract(r"(\d{4})")
contacts

### ✏️ 练习 13-1
变量 `s = "联系方式：13800001111，备用：13911112222"`，用 `re.findall` 提取出两个手机号（提示：手机号是 11 位连续数字，`\d{11}`）。

### ✏️ 练习 13-2
给 `contacts` 新增一列「电话前三位」，用 `str.extract` 或切片提取电话号码的前 3 位数字。

In [ ]:
# 练习 13-1：在这里写你的代码

In [ ]:
# 练习 13-2：在这里写你的代码

## Day 14 · 实战：文本清洗与规范输出

**任务**：把这份脏联系人数据彻底清洗干净，并为每人生成一行规范的档案描述。

清洗清单（照着实做一遍）：
1. 姓名：去两端空白、去掉名字中间的空格；
2. 城市：统一为首字母大写；
3. 电话：只保留数字；
4. 注册日期：统一为 `年-月-日` 格式；
5. 用 f-string 拼接成规范描述。

In [ ]:
import pandas as pd

# 重新造一份脏数据，从头开始清洗
contacts = pd.DataFrame({
    "姓名": ["  张三 ", "李 四", "王五", "赵六"],
    "城市": ["beijing", "SHANGHAI", "Guangzhou", "shenzhen"],
    "电话": ["138-0000-1111", "139 1111 2222", "137.2222.3333", "136-3333-4444"],
    "注册日期": ["2026-01-05", "2026/02/11", "2026.03.20", "2026-04-15"],
})

# 1. 姓名：去两端空白，再去掉中间空格（\s+ 表示所有连续空白）
contacts["姓名"] = contacts["姓名"].str.strip().str.replace(r"\s+", "", regex=True)

# 2. 城市：统一首字母大写
contacts["城市"] = contacts["城市"].str.title()

# 3. 电话：只保留数字
contacts["电话"] = contacts["电话"].str.replace(r"\D", "", regex=True)

# 4. 注册日期：把 / 和 . 统一成 -
contacts["注册日期"] = (contacts["注册日期"]
                      .str.replace("/", "-", regex=False)
                      .str.replace(".", "-", regex=False))
contacts

In [ ]:
# 5. 逐行拼接规范描述（apply 对每一行执行同一个函数）
def make_profile(row):
    return f"{row['姓名']}（{row['城市']}），电话：{row['电话']}，注册日期：{row['注册日期']}"

contacts["档案"] = contacts.apply(make_profile, axis=1)   # axis=1 表示按行处理

for line in contacts["档案"]:
    print(line)

### ✏️ 练习 14-1
修改 `make_profile` 函数，让档案描述变成：`【姓名】城市-电话` 的紧凑格式，例如 `【张三】Beijing-13800001111`，重新生成并打印。

In [ ]:
# 练习 14-1：在这里写你的代码

## Day 15 · 缓冲日：补漏 + 互动答疑 + 测验 3

**今天不新学内容，安排如下：**

1. **补漏（约 1 小时）**：把 Day 12–14 没完成的部分补齐；
2. **整理问题清单（约 15 分钟）**：翻看本 notebook 末尾的「我的问题清单」，已解决的标记掉，没解决的按主题归类；
3. **互动答疑（约 30 分钟）**：和老师 / 同学 / 辅导者约一次答疑，照着问题清单逐条问，当场把答案记进清单；
4. **完成测验 3**：打开网页版学习计划中的 <b>quiz3.html</b>（10 题，即时判分含解析，60 合格 / 80 优秀）。

> 💡 **平时怎么记录问题？** 每天学习中一遇到卡住的地方，立刻在问题清单里记一行：日期、做了什么、报错信息或困惑点。问题越具体，答疑效率越高——「groupby 之后取某一列报错 KeyError」比「groupby 不会」好问得多。

---

## 📒 我的问题清单（随手记录，缓冲日答疑前整理）

> 使用方法：学习中遇到任何卡住的地方，双击本单元格，在表格里加一行。答疑后把「状态」改为已解决并写下答案要点。

| 日期 | 遇到的问题 / 报错信息 | 状态（待问 / 已解决） | 答案要点 |
|---|---|---|---|
| 示例 | groupby 之后想取某一列却报错 KeyError | 已解决 | 结果是 Series，要用 reset_index() 还原成表 |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---

## ✅ 参考答案（先独立完成，再对照）

In [ ]:
# 练习 12-1
contacts["注册日期"] = (contacts["注册日期"]
                      .str.replace("/", "-", regex=False)
                      .str.replace(".", "-", regex=False))
contacts

In [ ]:
import re

# 练习 13-1
s = "联系方式：13800001111，备用：13911112222"
print(re.findall(r"\d{11}", s))

# 练习 13-2
contacts["电话前三位"] = contacts["电话"].str.extract(r"(\d{3})")
contacts[["电话", "电话前三位"]]

In [ ]:
# 练习 14-1
def make_profile_v2(row):
    return f"【{row['姓名']}】{row['城市']}-{row['电话']}"

contacts["档案"] = contacts.apply(make_profile_v2, axis=1)
for line in contacts["档案"]:
    print(line)

---

## 🎉 阶段三完成，下一站

1. 回到网页版学习计划，完成 **测验 3（quiz3.html）**；
2. 打开 [阶段四 · SQL 数据库](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage4_sql.ipynb)，进入 Day 16。